In [ ]:
# ============================================================
# CELL 1 — IMPORTS, CONFIGURATION & BACKUP
# ============================================================
# ALGORITHM NOTE
# ─────────────────────────────────────────────────────────────
# sklearn < 1.2 → default = SAMME.R (real-valued probabilities,
# faster convergence, usually better accuracy)
# sklearn ≥ 1.4 → SAMME.R removed; only SAMME is available.
# Since you get an error on algorithm='SAMME.R', you are on
# sklearn ≥ 1.4 — both notebooks effectively run SAMME.
# ─────────────────────────────────────────────────────────────

import os
import shutil
import datetime
import numpy as np
import joblib

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA PATHS ────────────────────────────────────────────────
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ───────────────────────────────────────────────
# Keep B-specific filename so B/C runs do not overwrite each other
MODEL_SAVE_PATH = r"C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost_mlp_B.joblib"
MODEL_BACKUP_DIR = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

# ── GLOBAL SETTINGS / HYPERPARAMETERS ────────────────────────
RANDOM_STATE = 42

os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== ADABOOST + MLP WEAK-LEARNER CONFIGURATION ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

if os.path.exists(MODEL_SAVE_PATH):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(
        MODEL_BACKUP_DIR,
        f"tissue_density_model_adaboost_mlp_B_{timestamp}.joblib"
    )
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f"Existing model backed up → {backup_path}")
else:
    print("No existing model found — fresh training run.")

In [ ]:
# ============================================================
# CELL 2 — LOAD & SPLIT (PATIENT-WISE, NO SMOTE)
# ============================================================
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter
import numpy as np

print("\nLoading feature matrix, labels, and groups...")
X = np.load(X_PATH)
y = np.load(Y_PATH)
# Load the groups file you created with label_assignment.py
groups = np.load(r"C:\Users\kalig\OneDrive\Desktop\train_groups.npy")

print(f"Loaded → X={X.shape}, y={y.shape}, groups={groups.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# ── SPLIT HYPERPARAMETERS ────────────────────────────────────
TEST_SIZE = 0.10
VAL_SIZE_OF_REMAINING = 0.1111111111  # ~= 10% of full data after 90/10 test split
RANDOM_STATE = 42

# Step 1: Hold out untouched Test Set based on PATIENTS
gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_temp, y_temp, groups_temp = X[train_val_idx], y[train_val_idx], groups[train_val_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Step 2: Split remaining data into Train + Validation based on PATIENTS
gss_val = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_OF_REMAINING, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_val.split(X_temp, y_temp, groups=groups_temp))

X_train, y_train = X_temp[train_idx], y_temp[train_idx]
X_val, y_val = X_temp[val_idx], y_temp[val_idx]

print(f"\nTraining distribution   : {Counter(y_train)}")
print(f"Validation distribution : {Counter(y_val)}")
print(f"Test distribution       : {Counter(y_test)}")

# Summary counts
print(f"\n{'='*60}")
print("DATA SUMMARY (NO SMOTE)")
print(f"{'='*60}")
print(f"Total training images : {X_train.shape[0]}")
print(f"Validation images     : {X_val.shape[0]}")
print(f"Test images           : {X_test.shape[0]}")
print(f"Features per image    : {X_train.shape[1]}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 3 — MLP WEAK-LEARNER WRAPPER (v2)
# ============================================================
# WHY A WRAPPER?
# ─────────────────────────────────────────────────────────────
# AdaBoostClassifier internally calls estimator.fit(X, y,
# sample_weight=w) to re-focus each round on hard samples.
# sklearn's MLPClassifier.fit() does NOT accept sample_weight,
# so a direct plug-in raises a TypeError. This wrapper converts
# the weight vector into a resampling distribution instead.
# ─────────────────────────────────────────────────────────────

class WeakMLP(BaseEstimator, ClassifierMixin):
    """
    Tiny MLP wrapper for AdaBoostClassifier.
    Handles sample_weight via weighted resampling.
    Adds L2 regularisation, early stopping, and explicit batch size.
    """

    def __init__(
        self,
        hidden_layer_sizes=(32,),
        max_iter=15,
        activation='relu',
        alpha=1e-3,
        batch_size=64,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        use_class_balance=False,
        random_state=None
    ):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.max_iter = max_iter
        self.activation = activation
        self.alpha = alpha
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.use_class_balance = use_class_balance
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        rng = np.random.RandomState(self.random_state)

        final_weights = np.ones(len(y), dtype=float)

        # 1) AdaBoost hard-sample emphasis
        if sample_weight is not None:
            boost_w = np.asarray(sample_weight, dtype=float)
            boost_w = boost_w / boost_w.sum()
            final_weights *= boost_w

        # 2) Optional inverse class-frequency emphasis
        if self.use_class_balance:
            class_counts = Counter(y)
            class_weight_map = {
                cls: len(y) / (len(class_counts) * count)
                for cls, count in class_counts.items()
            }
            class_w = np.array([class_weight_map[label] for label in y], dtype=float)
            final_weights *= class_w

        final_weights = final_weights / final_weights.sum()

        idx = rng.choice(len(X), size=len(X), replace=True, p=final_weights)
        X_r, y_r = X[idx], y[idx]

        self.mlp_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes,
            max_iter=self.max_iter,
            activation=self.activation,
            solver='adam',
            alpha=self.alpha,
            batch_size=self.batch_size,
            learning_rate_init=1e-3,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            n_iter_no_change=self.n_iter_no_change,
            warm_start=False,
            random_state=self.random_state
        )

        self.mlp_.fit(X_r, y_r)
        self.classes_ = self.mlp_.classes_
        return self

    def predict(self, X):
        check_is_fitted(self, 'mlp_')
        return self.mlp_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self, 'mlp_')
        return self.mlp_.predict_proba(X)

print("WeakMLP v2 defined — L2 alpha, early stopping, batch_size=64")
print("Architecture options:")
print(" A: (16,) ultra-weak")
print(" B: (32,) weak ← DEFAULT / best so far")
print(" C: (32, 16) mild")

In [ ]:
# ============================================================
# CELL 4 — GRID SEARCH (ARCH B ONLY) + PICK BEST MODEL
# ============================================================

# ── FIXED ARCHITECTURE ───────────────────────────────────────
ARCH = 'B'
ARCH_LABEL = "Weak (32,) ← fixed for this search"
HIDDEN_LAYER_SIZES = (32,)

# ── FIXED MLP SETTINGS ───────────────────────────────────────
MLP_ALPHA = 1e-3
MLP_BATCH_SIZE = 64
MLP_ACTIVATION = 'relu'
MLP_EARLY_STOPPING = True
MLP_VALIDATION_FRACTION = 0.1
MLP_N_ITER_NO_CHANGE = 5
USE_CLASS_BALANCE = False

# ── 9 SEARCH COMBINATIONS (MANUAL, NOT FULL 27-GRID) ─────────
SEARCH_COMBINATIONS = [
    {"MLP_MAX_ITER": 5, "N_ESTIMATORS": 100, "LEARNING_RATE": 0.3},
    {"MLP_MAX_ITER": 5, "N_ESTIMATORS": 150, "LEARNING_RATE": 0.5},
    {"MLP_MAX_ITER": 5, "N_ESTIMATORS": 250, "LEARNING_RATE": 0.8},
    {"MLP_MAX_ITER": 10, "N_ESTIMATORS": 100, "LEARNING_RATE": 0.5},
    {"MLP_MAX_ITER": 10, "N_ESTIMATORS": 150, "LEARNING_RATE": 0.8},
    {"MLP_MAX_ITER": 10, "N_ESTIMATORS": 250, "LEARNING_RATE": 0.3},
    {"MLP_MAX_ITER": 15, "N_ESTIMATORS": 100, "LEARNING_RATE": 0.8},
    {"MLP_MAX_ITER": 15, "N_ESTIMATORS": 150, "LEARNING_RATE": 0.3},
    {"MLP_MAX_ITER": 15, "N_ESTIMATORS": 250, "LEARNING_RATE": 0.5},
]

print(f"Selected architecture : {ARCH_LABEL}")

print(f"\n{'='*70}")
print("GRID SEARCH CONFIG")
print(f"{'='*70}")
print(f"ARCH : {ARCH}")
print(f"architecture : {HIDDEN_LAYER_SIZES}")
print(f"total combinations : {len(SEARCH_COMBINATIONS)}")
print(f"MLP alpha (L2) : {MLP_ALPHA}")
print(f"MLP batch_size : {MLP_BATCH_SIZE}")
print(f"MLP early_stopping : {MLP_EARLY_STOPPING}")
print(f"MLP validation_fraction : {MLP_VALIDATION_FRACTION}")
print(f"MLP n_iter_no_change : {MLP_N_ITER_NO_CHANGE}")
print(f"use_class_balance : {USE_CLASS_BALANCE}")
print(f"Algorithm : SAMME")
print(f"{'='*70}")

search_results = []
best_model = None
best_combo = None
best_val_acc = -1.0
chosen = {"label": ARCH_LABEL, "hidden_layer_sizes": HIDDEN_LAYER_SIZES}

for i, combo in enumerate(SEARCH_COMBINATIONS, start=1):
    MLP_MAX_ITER = combo["MLP_MAX_ITER"]
    N_ESTIMATORS = combo["N_ESTIMATORS"]
    LEARNING_RATE = combo["LEARNING_RATE"]

    print(f"\n[{i}/{len(SEARCH_COMBINATIONS)}] Training combination")
    print(f" MLP_MAX_ITER = {MLP_MAX_ITER}")
    print(f" N_ESTIMATORS = {N_ESTIMATORS}")
    print(f" LEARNING_RATE = {LEARNING_RATE}")

    base_estimator = WeakMLP(
        hidden_layer_sizes=HIDDEN_LAYER_SIZES,
        max_iter=MLP_MAX_ITER,
        activation=MLP_ACTIVATION,
        alpha=MLP_ALPHA,
        batch_size=MLP_BATCH_SIZE,
        early_stopping=MLP_EARLY_STOPPING,
        validation_fraction=MLP_VALIDATION_FRACTION,
        n_iter_no_change=MLP_N_ITER_NO_CHANGE,
        use_class_balance=USE_CLASS_BALANCE,
        random_state=RANDOM_STATE
    )

    candidate_model = AdaBoostClassifier(
        estimator=base_estimator,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        random_state=RANDOM_STATE
    )

    candidate_model.fit(X_train, y_train)

    y_val_pred_candidate = candidate_model.predict(X_val)
    val_acc_candidate = accuracy_score(y_val, y_val_pred_candidate)

    result = {
        "combo_no": i,
        "MLP_MAX_ITER": MLP_MAX_ITER,
        "N_ESTIMATORS": N_ESTIMATORS,
        "LEARNING_RATE": LEARNING_RATE,
        "val_accuracy": val_acc_candidate
    }
    search_results.append(result)

    print(f" Validation accuracy : {val_acc_candidate:.4f}")

    if val_acc_candidate > best_val_acc:
        best_val_acc = val_acc_candidate
        best_model = candidate_model
        best_combo = result

print(f"\n{'='*70}")
print("ALL COMBINATIONS — VALIDATION ACCURACY")
print(f"{'='*70}")
for r in search_results:
    print(
        f"#{r['combo_no']:>2} | "
        f"MLP_MAX_ITER={r['MLP_MAX_ITER']:<2} | "
        f"N_ESTIMATORS={r['N_ESTIMATORS']:<3} | "
        f"LEARNING_RATE={r['LEARNING_RATE']:<3} | "
        f"val_acc={r['val_accuracy']:.4f}"
    )

search_results_sorted = sorted(search_results, key=lambda x: x["val_accuracy"], reverse=True)

print(f"\n{'='*70}")
print("RANKED COMBINATIONS")
print(f"{'='*70}")
for rank, r in enumerate(search_results_sorted, start=1):
    print(
        f"Rank {rank:>2} | "
        f"MLP_MAX_ITER={r['MLP_MAX_ITER']:<2} | "
        f"N_ESTIMATORS={r['N_ESTIMATORS']:<3} | "
        f"LEARNING_RATE={r['LEARNING_RATE']:<3} | "
        f"val_acc={r['val_accuracy']:.4f}"
    )

print(f"\n{'='*70}")
print("BEST COMBINATION SELECTED")
print(f"{'='*70}")
print(f"ARCH : {ARCH}")
print(f"architecture : {HIDDEN_LAYER_SIZES}")
print(f"MLP_MAX_ITER : {best_combo['MLP_MAX_ITER']}")
print(f"N_ESTIMATORS : {best_combo['N_ESTIMATORS']}")
print(f"LEARNING_RATE : {best_combo['LEARNING_RATE']}")
print(f"Best validation accuracy : {best_combo['val_accuracy']:.4f}")
print(f"{'='*70}")

model = best_model
MLP_MAX_ITER = best_combo["MLP_MAX_ITER"]
N_ESTIMATORS = best_combo["N_ESTIMATORS"]
LEARNING_RATE = best_combo["LEARNING_RATE"]

In [ ]:
# ============================================================
# CELL 5 — EVALUATE BEST MODEL ON VALIDATION + TEST
# ============================================================

# ── EVALUATION / REPORTING HYPERPARAMETERS ───────────────────
SHOW_FIRST_N_PROB_ROWS = 5
RARE_CLASS_NAMES = [
    'Tissue Density Category 1',
    'Tissue Density Category 4'
]
TARGET_NAMES = [
    'Tissue Density Category 1',
    'Tissue Density Category 2',
    'Tissue Density Category 3',
    'Tissue Density Category 4'
]

# ── PREDICT ON VALIDATION AND TEST ───────────────────────────
print("\nEvaluating BEST selected model on validation set...")
y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)

print("Evaluating BEST selected model on held-out test set...")
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

# ── METRICS ──────────────────────────────────────────────────
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

val_cm = confusion_matrix(y_val, y_val_pred)
test_cm = confusion_matrix(y_test, y_test_pred)

val_report_dict = classification_report(
    y_val,
    y_val_pred,
    target_names=TARGET_NAMES,
    zero_division=0,
    output_dict=True
)

test_report_dict = classification_report(
    y_test,
    y_test_pred,
    target_names=TARGET_NAMES,
    zero_division=0,
    output_dict=True
)

# ── SUMMARY REPORT ───────────────────────────────────────────
print(f"\n{'='*60}")
print("PERFORMANCE SUMMARY")
print(f"{'='*60}")
print(f"Architecture : {chosen['label']}")
print(f"Best combo selected from validation search:")
print(f" MLP_MAX_ITER : {MLP_MAX_ITER}")
print(f" N_ESTIMATORS : {N_ESTIMATORS}")
print(f" LEARNING_RATE : {LEARNING_RATE}")
print(f" MLP alpha (L2) : {MLP_ALPHA}")
print(f" MLP batch_size : {MLP_BATCH_SIZE}")
print(f"Training images : {len(X_train)}")     # <--- UPDATED
print(f"Validation images : {len(X_val)}")   # <--- UPDATED
print(f"Test images : {len(X_test)}")        # <--- UPDATED
print(f"Validation accuracy : {val_acc:.4f}")
print(f"Test accuracy : {test_acc:.4f}")
print(f"{'='*60}")

# ── VALIDATION REPORT ────────────────────────────────────────
print(f"\n{'='*60}")
print("VALIDATION REPORT")
print(f"{'='*60}")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=TARGET_NAMES,
    zero_division=0
))

print("Validation Confusion Matrix:")
header = " " + " ".join(f"Pred {i+1:>4}" for i in range(4))
print(header)
for i, row in enumerate(val_cm):
    print(f"Actual {i+1} " + " ".join(f"{v:>8d}" for v in row))

print(f"\nFirst {SHOW_FIRST_N_PROB_ROWS} validation class-probability rows:")
print(y_val_proba[:SHOW_FIRST_N_PROB_ROWS])

print(f"\n{'='*60}")
print("VALIDATION RARE-CLASS FOCUS")
print(f"{'='*60}")
for name in RARE_CLASS_NAMES:
    r = val_report_dict[name]
    print(f"{name}")
    print(f" recall : {r['recall']:.4f}")
    print(f" f1-score : {r['f1-score']:.4f}")
    print(f" precision : {r['precision']:.4f}")
print(f"{'='*60}")

# ── TEST REPORT ──────────────────────────────────────────────
print(f"\n{'='*60}")
print("TEST REPORT")
print(f"{'='*60}")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=TARGET_NAMES,
    zero_division=0
))

print("Test Confusion Matrix:")
print(header)
for i, row in enumerate(test_cm):
    print(f"Actual {i+1} " + " ".join(f"{v:>8d}" for v in row))

print(f"\nFirst {SHOW_FIRST_N_PROB_ROWS} test class-probability rows:")
print(y_test_proba[:SHOW_FIRST_N_PROB_ROWS])

print(f"\n{'='*60}")
print("TEST RARE-CLASS FOCUS")
print(f"{'='*60}")
for name in RARE_CLASS_NAMES:
    r = test_report_dict[name]
    print(f"{name}")
    print(f" recall : {r['recall']:.4f}")
    print(f" f1-score : {r['f1-score']:.4f}")
    print(f" precision : {r['precision']:.4f}")
print(f"{'='*60}")

# Keep names expected by later cells if needed
y_pred = y_test_pred
y_proba = y_test_proba

In [ ]:
# ============================================================
# CELL 6 — CLASS CONFIDENCE SNAPSHOT
# ============================================================
print("\nSample class-probability breakdown (first 10 test images):\n")

header = (
    f"{'Image':<8} | "
    f"{'Cat 1':>6} {'Cat 2':>6} {'Cat 3':>6} {'Cat 4':>6} | "
    f"{'Pred':>5} {'True':>5}"
)
print(header)
print("-" * len(header))

for i in range(min(10, len(X_test))):
    p_row = y_proba[i]
    pred  = y_pred[i]
    true  = y_test[i]
    match = "✅" if pred == true else "❌"

    print(
        f"{i+1:<8} | "
        f"{p_row[0]:>6.3f} {p_row[1]:>6.3f} {p_row[2]:>6.3f} {p_row[3]:>6.3f} | "
        f"{pred:>5} {true:>5}  {match}"
    )

In [ ]:
# ============================================================
# CELL 7 — SAVE MODEL
# ============================================================
print(f"\nSaving model → {MODEL_SAVE_PATH}")
joblib.dump(model, MODEL_SAVE_PATH)
print(" Done! AdaBoost + MLP tissue density model saved.")

In [ ]:
# ============================================================
# CELL 8 — PREDICTION UTILITY
# ============================================================
def predict_new(X_new: np.ndarray):
    """
    Run inference on new feature vectors.

    Parameters
    ----------
    X_new : np.ndarray, shape (n_samples, n_features)

    Returns
    -------
    preds       : predicted tissue density class labels
    class_probs : per-class probabilities from the MLP ensemble
    """
    preds       = model.predict(X_new)
    class_probs = model.predict_proba(X_new)
    return preds, class_probs

print(" Prediction utility ready.")
print("Usage:")
print("  preds, class_probs = predict_new(X_new)")
print("\nOr load from disk:")
print(f"  model = joblib.load(r'{MODEL_SAVE_PATH}')")
print("  preds, class_probs = model.predict(X_new), model.predict_proba(X_new)")